In [1]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

sys.path.append(os.path.abspath(".."))
from diffuser.diffuser_ddpm_linear_schedule import Diffuser_DDPM_linear_schedule 
from diffuser.unet import DiffusionUNet

from dataset_loader import load_dataset
from torch.utils.data import DataLoader
from vae.vae import VAE

# Config:
from diffuser.unet_config import DiffuserConfig

c:\Users\jaspe\Documents\Github_projects\Generative_Modelling_Project\DSAIT4030-group12\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setup

In [2]:
#Init
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

torch.manual_seed(0)
np.random.seed(0)

cpu


In [3]:
# Create scheduler model
ddpm_model = Diffuser_DDPM_linear_schedule(total_timesteps=1000, beta_start=0.0001, beta_end=0.02)

ddpm_model.betas = ddpm_model.betas.to(device)
ddpm_model.alphas = ddpm_model.alphas.to(device)
ddpm_model.alpha_bars = ddpm_model.alpha_bars.to(device)

In [4]:
# Create model
config = DiffuserConfig()
unet = DiffusionUNet(
    config=config,
    model_in_channels=3,
    model_out_channels=3
).to(device)

optimizer = torch.optim.AdamW(unet.parameters(), lr=2e-4, weight_decay=1e-5)

In [6]:
# Load VAE
vae = VAE(mode="kl").to(device)
checkpoint = torch.load("../step_100000.pt", map_location=device, weights_only=False)

vae.load_state_dict(checkpoint["vae"], strict=False)

<All keys matched successfully>

In [7]:
# Loads data and uses VAE to generate latents
def load_data(NUM_WORKERS, BATCH_SIZE, DATASET):
    train_set = load_dataset(DATASET, split="train")

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    return train_loader

In [8]:
def train_step(data, minibatch_size, model, ddpm):
    
    model.train()
    optimizer.zero_grad(set_to_none=True)

    # Number of iterations per training step equals the amount of minibatches needed to sample all data per training step.
    number_of_minibatches = data.shape[0] // minibatch_size
    total_loss = 0.0

    for i in range(number_of_minibatches):
        
        # Retrieve the data corresponding to the current minibatch.
        start_idx = i * minibatch_size
        end_idx = start_idx + minibatch_size
        data_minibatch = data[start_idx:end_idx]

        data_minibatch = vae.encode(data_minibatch)

        # Randomly sample diffusion timesteps for each data sample in the minibatch.
        B = data_minibatch.shape[0]
        t = torch.randint(0, ddpm.total_timesteps, (B,), device=device, dtype=torch.long).view(-1)

        # Perform a forward diffusion step to time t and retrieve the corresponding noisy sample x_t and the true added noise.
        x_t, true_noise = ddpm.forward_diffusion(data_minibatch, t)
        
        # Let the prediction model predict the added noise given the noisy sample x_t and the diffusion timestep t.
        pred_noise = model(x_t, t)
        
        # Compute the loss as the mean squared error and normalize based on the number of minibatches.
        loss = F.mse_loss(pred_noise, true_noise) / number_of_minibatches
        
        # Perform backward pass to compute gradient per minibatch.
        loss.backward()

        total_loss += loss.detach().item()
        
    # Perform an optimization step based on the accumulated gradients of all minibatches.
    optimizer.step()

    return total_loss

# Sampling

In [9]:
@torch.no_grad()
def sample(model, ddpm, shape, fixed_noise=None):
    model.eval()

    x = torch.randn(shape, device=device)

    if fixed_noise != None:
        x = fixed_noise

    for t in range(ddpm.total_timesteps - 1, 0, -1):
        ts = torch.full((shape[0],), t, device=device, dtype=torch.long).view(-1)
        pred_noise = model(x, ts)
        x = ddpm.reverse_diffusion(x, ts, pred_noise)

    return x

In [10]:
def show_images(img_batch, title=None):
    img_batch = img_batch.detach().cpu()
    fig, axes = plt.subplots(1, img_batch.shape[0], figsize=(4 * img_batch.shape[0], 4))
    if img_batch.shape[0] == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        img = img_batch[i]
        img = (img + 1.0) / 2.0
        img = (img * 255.0).clamp(0, 255).byte()
        img = img.permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.axis("off")

    if title:
        fig.suptitle(title)
    plt.show()

# Run and Show

In [11]:
# training loop:
num_steps = 10000
batch_size = 512
minibatch_size = 128
num_workers = 0
loss_history = []

fixed_noise = torch.randn(1, 3, 64, 64, device=device)
print_loss_every = 1
preview_every = 100

for step in range(num_steps):
    data = load_data(num_workers, batch_size, "celeba")
    loss = train_step(data, minibatch_size, unet, ddpm_model)
    loss_history.append(loss)

    if step % print_loss_every == 0:
        print(f"step {step:5d} | loss {loss:.4f}")

    if step % preview_every == 0:
        unet.eval()
        with torch.no_grad():
            samples = sample(unet, ddpm_model, (1, 3, 64, 64))
        show_images(samples, title=f"step {step}")
        unet.train()

c:\Users\jaspe\Documents\Github_projects\Generative_Modelling_Project
Saving celeba to disk for faster loading...


c:\Users\jaspe\Documents\Github_projects\Generative_Modelling_Project\DSAIT4030-group12\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jaspe\.cache\huggingface\hub\datasets--korexyz--celeba-hq-256x256. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Saving the dataset (1/1 shards): 100%|██████████

Done! Saved to c:\Users\jaspe\Documents\Github_projects\Generative_Modelling_Project\data\celeba


AttributeError: 'DataLoader' object has no attribute 'shape'

In [ ]:
show_images(samples, title="Generated samples")

In [ ]:
samples = sample(unet, ddpm_model, (1, 3, 64, 64))